# Moving Health — previsão contextual e simulação de demanda

Este notebook contém dois experimentos separados:

1. **Dados reais:** previsão mensal da quantidade aprovada de procedimentos ambulatoriais do SIA/SUS por município.
2. **Dados sintéticos:** simulação horária de solicitações, capacidade hipotética e comparação de cenários com Pandas, NumPy, scikit-learn e Matplotlib.

A simulação mostra como uma previsão poderia aparecer na plataforma, mas seus dados e métricas não representam viagens observadas do Moving Health.

## 1. Problema de negócio e escolha do primeiro problema

O Moving Health conectará pacientes a unidades de saúde e também poderá conectar profissionais licenciados a pacientes em domicílio. Ainda não há histórico operacional próprio confirmado.

| Possibilidade | Dados hoje | Complexidade | Decisão nesta etapa |
|---|---|---|---|
| Demanda de viagens por região e horário | Não há pedidos do Moving Health; há dados públicos assistenciais que são apenas um proxy | Média a alta | Adiada como previsão de corridas; usamos a produção assistencial apenas como contexto |
| Tempo estimado de chegada (ETA) | Exigiria origens, destinos, horários e trajetos reais | Alta | Adiada até haver dados de viagens e integração com roteamento |
| Matching de motoristas/profissionais | Exigiria disponibilidade, localização, credenciamento e desfechos de atendimento | Alta | Adiada até existir cadastro operacional minimizado e regras de negócio |
| Disponibilidade de atendimento | Exigiria agenda, capacidade e disponibilidade atualizada das unidades/profissionais | Média a alta | Adiada até integrar fontes confiáveis de capacidade |

**Problema escolhido para esta primeira demonstração:** prever a produção ambulatorial aprovada por município e mês. A escolha aproveita uma série pública real já presente no repositório e permite demonstrar preparação temporal, treino, comparação com baseline e uma interface de previsão. Ela não equivale à previsão operacional final de viagens.

Uma segunda trilha, separada da base pública, gera solicitações horárias sintéticas. Ela permite demonstrar uma previsão de contagens e simular o efeito de demanda e frota em gráficos e tabelas, sem confundir essa experiência com dados observados.

## 2. Fonte de dados e procedência

Fonte: SIA/SUS TABNET/DATASUS, consulta para o estado de São Paulo, por município de atendimento × mês de atendimento, conteúdo Qtd.aprovada. A extração preservada no projeto cobre janeiro de 2008 a agosto de 2026.

A pasta data/raw mantém o CSV retornado e a resposta HTML recebida. A origem, o formato, a semântica, a data de acesso e as limitações estão detalhados em docs/data_sources.md.

- **Real:** produção ambulatorial agregada do SIA/SUS.
- **Derivado:** tabela longa município, mês e quantidade aprovada, gerada pelo treinamento.
- **Simulado:** nenhum dado simulado é usado neste experimento.
- **Limitação:** quantidade aprovada pode mudar por oferta, registro e processamento; não representa indivíduos nem deslocamentos.
- **Licença:** os termos de redistribuição ainda precisam ser confirmados antes de republicar os arquivos.

### Execução no Google Colab

Envie este notebook ao Colab. Na primeira célula de código, selecione o arquivo ZIP portátil do projeto. O notebook o extrai no ambiente temporário do Colab, encontra os módulos e o CSV pelo nome relativo e não depende de caminhos da máquina local. Ao final, baixe o pacote de resultados. Se a sessão do Colab reiniciar, envie novamente o ZIP do projeto.

O experimento de mobilidade é sintético e gerado pelo projeto. Não foi usado um CSV público de táxis: ele não forneceria rótulos de viagens de saúde do Moving Health. O CSV sintético, as hipóteses e os resultados ficam separados da extração real.

In [1]:
from pathlib import Path
import csv
import json
import sys
import zipfile

def localizar_raiz(procurar_em):
    if (procurar_em / "scripts" / "train_sia_forecast.py").is_file():
        return procurar_em
    for arquivo in procurar_em.rglob("train_sia_forecast.py"):
        if arquivo.parent.name == "scripts":
            return arquivo.parent.parent
    return None

candidatos = [Path.cwd(), *Path.cwd().parents]
ROOT = next((raiz for raiz in candidatos if (raiz / "scripts" / "train_sia_forecast.py").is_file()), None)

try:
    from google.colab import files as colab_files
except ImportError:
    colab_files = None

if ROOT is None:
    if colab_files is None:
        raise RuntimeError(
            "Projeto não encontrado. No Google Colab, envie o ZIP portátil do projeto."
        )
    enviados = colab_files.upload()
    zips = [nome for nome in enviados if nome.lower().endswith(".zip")]
    if len(zips) != 1:
        raise ValueError("Envie exatamente um arquivo ZIP do projeto Moving Health.")
    pacote = Path.cwd() / zips[0]
    pacote.write_bytes(enviados[zips[0]])
    destino = Path.cwd() / "moving_health_project"
    destino.mkdir(parents=True, exist_ok=True)
    base_segura = destino.resolve()
    with zipfile.ZipFile(pacote) as arquivo_zip:
        for membro in arquivo_zip.infolist():
            caminho_extraido = (destino / membro.filename).resolve()
            if caminho_extraido != base_segura and base_segura not in caminho_extraido.parents:
                raise ValueError("O ZIP contém um caminho de arquivo inválido.")
        arquivo_zip.extractall(destino)
    ROOT = localizar_raiz(destino)
    if ROOT is None:
        raise FileNotFoundError(
            "O ZIP não contém scripts/train_sia_forecast.py."
        )

sys.path.insert(0, str(ROOT / "scripts"))
from sia_tabnet_data import ler_tabela, rotulos_meses, parsear_contagem
from train_sia_forecast import (
    SAIDA_MODELO,
    SAIDA_PROCESSADA,
    SAIDA_RELATORIOS,
    executar,
    ler_painel,
)

ARQUIVO_BRUTO = ROOT / "data/raw/sia_sus_sp/tabnet_qasp_municipio_mes_qtd_aprovada.csv"
if not ARQUIVO_BRUTO.is_file():
    raise FileNotFoundError(
        "CSV do SIA/SUS não encontrado no pacote do projeto."
    )
print("Projeto Moving Health encontrado.")
print("CSV bruto encontrado pelo caminho relativo.")

Projeto Moving Health encontrado.
CSV bruto encontrado pelo caminho relativo.


## Dependências para simulação no Colab

Os scripts da trilha SIA/SUS usam a biblioteca padrão. Para a simulação, o notebook verifica se Pandas, NumPy, scikit-learn, Matplotlib e joblib estão disponíveis; instala o arquivo requirements-colab.txt somente se algum módulo estiver faltando e registra as versões usadas.

In [ ]:
import importlib.metadata
import importlib.util
import re
import subprocess
import sys

modulos_necessarios = {
    "pandas": "pandas",
    "numpy": "numpy",
    "sklearn": "scikit-learn",
    "matplotlib": "matplotlib",
    "joblib": "joblib",
}
faixas_versao = {
    "pandas": ((2, 2, 0), 4),
    "numpy": ((1, 26, 0), 3),
    "scikit-learn": ((1, 4, 0), 2),
    "matplotlib": ((3, 8, 0), 4),
    "joblib": ((1, 3, 0), 2),
}

def versao_numerica(texto):
    partes = []
    for parte in texto.split("."):
        digitos = re.match(r"\d+", parte)
        if not digitos:
            break
        partes.append(int(digitos.group()))
    return tuple((partes + [0, 0, 0])[:3])

faltantes_ou_incompativeis = []
for modulo, distribuicao in modulos_necessarios.items():
    try:
        instalado = importlib.metadata.version(distribuicao)
    except importlib.metadata.PackageNotFoundError:
        faltantes_ou_incompativeis.append(distribuicao)
        continue
    minimo, major_maximo = faixas_versao[distribuicao]
    if (
        importlib.util.find_spec(modulo) is None
        or versao_numerica(instalado) < minimo
        or versao_numerica(instalado)[0] >= major_maximo
    ):
        faltantes_ou_incompativeis.append(distribuicao)

if faltantes_ou_incompativeis:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "-r",
            str(ROOT / "requirements-colab.txt"),
        ]
    )

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import PoissonRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_poisson_deviance,
    mean_squared_error,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

runtime_versions = {
    "pandas": importlib.metadata.version("pandas"),
    "numpy": importlib.metadata.version("numpy"),
    "scikit_learn": importlib.metadata.version("scikit-learn"),
    "matplotlib": importlib.metadata.version("matplotlib"),
    "joblib": importlib.metadata.version("joblib"),
}
print("Versões do runtime:")
for nome, versao in runtime_versions.items():
    print(f"  {nome}: {versao}")

## 3. Inspeção e análise exploratória

A tabela original vem em formato largo: uma linha por município e uma coluna por mês. O leitor central do projeto valida a sequência mensal, converte o marcador TABNET “-” em zero, exclui a linha total do painel e rejeita valores numéricos irregulares. Células realmente vazias seriam contabilizadas e não seriam imputadas.

In [2]:
meses, series, ausencias = ler_painel(ARQUIVO_BRUTO)
quantidade_zeros = sum(
    valor == 0
    for valores in series.values()
    for valor in valores
)
top_municipios = sorted(
    ((sum(valores), municipio) for municipio, valores in series.items()),
    reverse=True,
)[:5]

print(f"Municípios com série completa: {len(series)}")
print(f"Meses: {len(meses)} ({meses[0]} a {meses[-1]})")
print(f"Valores zero convertidos do marcador TABNET: {quantidade_zeros}")
print(f"Municípios com células vazias: {len(ausencias)}")
print("Maiores totais acumulados no período (apenas para inspeção):")
for total, municipio in top_municipios:
    print(f"  {municipio}: {total:,}".replace(",", "."))

Municípios com série completa: 645
Meses: 224 (2008/Jan a 2026/Ago)
Valores zero convertidos do marcador TABNET: 1280
Municípios com células vazias: 0
Maiores totais acumulados no período (apenas para inspeção):
  355030 SAO PAULO: 6.274.242.550
  350950 CAMPINAS: 953.472.003
  355220 SOROCABA: 581.620.245
  354980 SAO JOSE DO RIO PRETO: 545.563.070
  354340 RIBEIRAO PRETO: 527.331.291


## 4. Preparação e atributos

A tabela derivada fica em formato longo e contém município, mês e alvo. O marcador “-” tem semântica de zero na legenda do TABNET; valores vazios não são inventados. A coluna Total não entra como série municipal.

O modelo candidato usa regressão linear por mínimos quadrados. Os atributos são uma tendência temporal em anos e efeitos fixos para os meses de janeiro a novembro; dezembro é o mês de referência. Não há normalização porque a escala é simples e as entradas são tempo e indicadores categóricos. A série é ajustada separadamente para cada município com histórico suficiente.

Essa opção usa somente Python padrão, mantendo o ambiente reproduzível sem instalar pacotes. Pandas, NumPy e scikit-learn poderiam simplificar operações e experimentos, mas não são necessários para este modelo pequeno e transparente; sua inclusão deverá ser justificada quando houver uma necessidade técnica concreta.

## 5. Separação temporal e baseline

Os últimos 12 meses disponíveis (setembro de 2025 a agosto de 2026) formam o holdout. Os meses anteriores são usados no treino; não embaralhamos observações temporais.

O baseline sazonal repete o valor observado 12 meses antes. Ele é uma referência forte e fácil de explicar para dados mensais. O modelo candidato só deve ser promovido à integração se melhorar a avaliação temporal.

In [3]:
from contextlib import redirect_stdout
from io import StringIO

registro_treino = StringIO()
with redirect_stdout(registro_treino):
    executar(
        entrada=ARQUIVO_BRUTO,
        saida_processada=SAIDA_PROCESSADA,
        saida_relatorios=SAIDA_RELATORIOS,
        horizonte_teste=12,
        horizonte_futuro=12,
        minimo_treino=36,
        saida_modelo=SAIDA_MODELO,
    )

print("Treinamento, métricas, previsões e artefato do modelo foram gerados.")

Treinamento, métricas, previsões e artefato do modelo foram gerados.


## 6. Treinamento, avaliação e interpretação

MAE e RMSE são erros em unidades de procedimentos aprovados. WAPE divide a soma dos erros absolutos pela soma dos valores observados e resume a qualidade agregada do painel.

O resultado é evidência do período de holdout e dos municípios desta extração, não uma garantia de desempenho futuro. Como os municípios têm escalas muito diferentes, a agregação dá mais peso, em termos absolutos, aos maiores volumes. A avaliação também é influenciada por mudanças de oferta e registro.

In [4]:
with (SAIDA_RELATORIOS / "metricas_avaliacao.csv").open(
    encoding="utf-8-sig", newline=""
) as arquivo:
    metricas = list(csv.DictReader(arquivo))

for linha in metricas:
    print(
        f"{linha['modelo']}: "
        f"MAE={float(linha['mae']):,.2f}; "
        f"RMSE={float(linha['rmse']):,.2f}; "
        f"WAPE={float(linha['wape']):.2%}; "
        f"R²={float(linha['r2']):.6f}; "
        f"observações={linha['observacoes']}; "
        f"municípios={linha['municipios']}"
    )

baseline_sazonal_12_meses: MAE=23,511.19; RMSE=118,449.14; WAPE=11.57%; R²=0.993944; observações=7740; municípios=645
tendencia_linear_sazonalidade_mensal: MAE=34,368.34; RMSE=145,817.43; WAPE=16.91%; R²=0.990823; observações=7740; municípios=645


**Decisão com base na validação:** o baseline sazonal vence o modelo de tendência nos três indicadores. Por isso, ele é o método recomendado pela API de demonstração. A regressão linear treinada continua exportada como candidata experimental, mas não é apresentada como superior nem adequada para decisões operacionais. O próximo passo metodológico é entender essa diferença com validação por janelas temporais e acompanhar estabilidade em novas extrações.

## 7. Previsão que aparece na solução

O serviço local disponibiliza a previsão agregada por município. Abaixo está um exemplo para o município de São Paulo, usando o método recomendado no holdout. O valor é uma estimativa de procedimentos aprovados e não de corridas.

In [5]:
with (SAIDA_RELATORIOS / "previsoes_futuras.csv").open(
    encoding="utf-8-sig", newline=""
) as arquivo:
    previsoes_sp = [
        linha
        for linha in csv.DictReader(arquivo)
        if linha["municipio"] == "355030 SAO PAULO"
    ]

print("Município: 355030 SAO PAULO")
for linha in previsoes_sp[:3]:
    valor = int(linha["previsao_recomendada"])
    print(f"{linha['mes']}: {valor:,} procedimentos aprovados estimados".replace(",", "."))

Município: 355030 SAO PAULO
2026-09: 40.435.558 procedimentos aprovados estimados
2026-10: 40.511.246 procedimentos aprovados estimados
2026-11: 35.280.501 procedimentos aprovados estimados


## 8. Exportação e integração por API

O treinamento salva o modelo candidato e seus coeficientes em models/moving_health_sia_sus.json. O mesmo artefato inclui o método recomendado pela avaliação, métricas e as últimas observações necessárias ao baseline.

Para iniciar a API local, use um segundo terminal na raiz do repositório:

~~~powershell
python scripts/forecast_service.py
~~~

Exemplo de consulta:

~~~text
GET http://127.0.0.1:8000/previsao?municipio=S%C3%A3o%20Paulo&horizonte=3
~~~

A API aceita o nome do município sem acento ou o código municipal presente na base. O endpoint devolve JSON com o método, a métrica de holdout, meses previstos e o aviso de interpretação. Este serviço é uma demonstração local; ainda não é um serviço de produção nem está ligado ao aplicativo completo.

In [6]:
from forecast_service import carregar_artifact, gerar_previsao

artifact = carregar_artifact(SAIDA_MODELO)
demonstracao = gerar_previsao(
    artifact=artifact,
    municipio="São Paulo",
    horizonte=3,
    estrategia="selecionado",
)
print(f"Método retornado: {demonstracao['modelo']}")
for item in demonstracao["previsoes"]:
    print(f"{item['mes']}: {item['qtd_aprovada_estimada']:,}".replace(",", "."))
print(f"Aviso: {demonstracao['aviso']}")

Método retornado: baseline_sazonal_12_meses
2026-09: 40.435.558
2026-10: 40.511.246
2026-11: 35.280.501
Aviso: Não representa pessoas, solicitações de transporte, viagens, corridas ou necessidade clínica.


## 9. Simulação sintética de pedidos por hora

O CSV sintético tem uma linha por hora, zona genérica e serviço. O gerador usa semente 2026, fatores horários e Gamma-Poisson para criar variação nas contagens. Veículos e capacidade também são simulados. A hipótese de capacidade é uma solicitação por veículo por hora.

Todos os registros têm a marca is_synthetic. As zonas não correspondem a bairros reais; as taxas são parâmetros didáticos, não estimativas de demanda.

In [ ]:
from simulate_trip_demand import (
    FORMA_DISTRIBUICAO,
    SEMENTE_PADRAO,
    gerar_dados_sinteticos,
)

CSV_SINTETICO = ROOT / "data/synthetic/moving_health_pedidos_sinteticos.csv"
if not CSV_SINTETICO.is_file():
    dados_gerados = gerar_dados_sinteticos(dias=90, semente=SEMENTE_PADRAO)
    CSV_SINTETICO.parent.mkdir(parents=True, exist_ok=True)
    dados_gerados.to_csv(CSV_SINTETICO, index=False, encoding="utf-8-sig")

df_sintetico = pd.read_csv(
    CSV_SINTETICO,
    sep=",",
    encoding="utf-8-sig",
    parse_dates=["timestamp"],
)
colunas_obrigatorias = {
    "timestamp", "zone_id", "service_type", "trips_requested",
    "vehicles_available", "trips_served", "unmet_requests", "is_synthetic",
}
faltantes_csv = colunas_obrigatorias - set(df_sintetico.columns)
if faltantes_csv:
    raise ValueError(f"Colunas ausentes no CSV sintético: {sorted(faltantes_csv)}")
if not df_sintetico["is_synthetic"].astype(str).str.lower().isin(["true", "1"]).all():
    raise ValueError("O CSV precisa identificar todos os registros como sintéticos.")
if df_sintetico["timestamp"].isna().any():
    raise ValueError("Há timestamps inválidos no CSV sintético.")

print(f"Arquivo sintético: {CSV_SINTETICO.name}")
print(f"Linhas: {len(df_sintetico):,}; intervalo: {df_sintetico['timestamp'].min()} a {df_sintetico['timestamp'].max()}")
print(f"Distribuição usada: {FORMA_DISTRIBUICAO}; semente: {SEMENTE_PADRAO}")
display(
    df_sintetico[[
        "timestamp", "zone_id", "service_type", "trips_requested",
        "vehicles_available", "trips_served", "unmet_requests", "is_synthetic",
    ]].head()
)
print(
    "Totais do cenário-base:",
    {
        "solicitacoes_sinteticas": int(df_sintetico["trips_requested"].sum()),
        "atendidas_pela_capacidade": int(df_sintetico["trips_served"].sum()),
        "nao_atendidas": int(df_sintetico["unmet_requests"].sum()),
    },
)

### Leitura visual da demanda sintética

O primeiro gráfico resume a contagem diária. O mapa de calor mostra a média simulada por dia da semana e hora. Eles visualizam as regras do gerador; não descrevem um padrão observado na cidade.

In [ ]:
RELATORIO_SINTETICO = ROOT / "reports/synthetic"
RELATORIO_SINTETICO.mkdir(parents=True, exist_ok=True)

demanda_diaria = (
    df_sintetico.groupby("timestamp")["trips_requested"]
    .sum()
    .resample("D")
    .sum()
)
demanda_hora_dia = df_sintetico.pivot_table(
    index="weekday",
    columns="hour",
    values="trips_requested",
    aggfunc="mean",
)

fig, eixos = plt.subplots(1, 2, figsize=(15, 4))
demanda_diaria.plot(ax=eixos[0], color="#8b0b3d", linewidth=1.5)
eixos[0].set_title("Solicitações sintéticas por dia")
eixos[0].set_xlabel("Data sintética")
eixos[0].set_ylabel("Solicitações")
eixos[0].grid(alpha=0.25)

imagem = eixos[1].imshow(demanda_hora_dia, aspect="auto", cmap="magma")
eixos[1].set_title("Média simulada: dia da semana × hora")
eixos[1].set_xlabel("Hora")
eixos[1].set_ylabel("Dia da semana (0=segunda)")
eixos[1].set_xticks(range(0, 24, 2))
fig.colorbar(imagem, ax=eixos[1], label="Solicitações por zona/serviço")
fig.tight_layout()
fig.savefig(RELATORIO_SINTETICO / "demanda_sintetica_eda.png", dpi=150, bbox_inches="tight")
plt.show()

## 10. Modelo de contagem e corte temporal

O alvo é trips_requested. As features são zona, serviço, hora, dia da semana e fim de semana; valores de pedidos e capacidade do próprio período de teste não entram no treino.

O holdout usa os últimos 14 dias inteiros. O baseline calcula, somente no treino, a média para a mesma combinação de zona, serviço, dia da semana e hora. O candidato é PoissonRegressor, um modelo de contagem com link logarítmico. Comparamos MAE, RMSE, WAPE e desvio Poisson. Métricas e previsões avaliam apenas esta distribuição sintética.

In [ ]:
def adicionar_features_temporais(dados):
    resultado = dados.copy()
    resultado["hour_sin"] = np.sin(2 * np.pi * resultado["hour"] / 24)
    resultado["hour_cos"] = np.cos(2 * np.pi * resultado["hour"] / 24)
    resultado["weekday_sin"] = np.sin(2 * np.pi * resultado["weekday"] / 7)
    resultado["weekday_cos"] = np.cos(2 * np.pi * resultado["weekday"] / 7)
    return resultado

df_modelo = adicionar_features_temporais(
    df_sintetico.sort_values("timestamp").reset_index(drop=True)
)
inicio_holdout = df_modelo["timestamp"].max() - pd.Timedelta(days=14) + pd.Timedelta(hours=1)
treino = df_modelo[df_modelo["timestamp"] < inicio_holdout].copy()
teste = df_modelo[df_modelo["timestamp"] >= inicio_holdout].copy()
if treino.empty or teste.empty:
    raise ValueError("Não foi possível montar o corte temporal de 14 dias.")

colunas_categoricas = ["zone_id", "service_type"]
colunas_numericas = [
    "hour_sin", "hour_cos", "weekday_sin", "weekday_cos", "is_weekend",
]
features = colunas_categoricas + colunas_numericas
alvo = "trips_requested"

preparador = ColumnTransformer(
    transformers=[
        ("categorias", OneHotEncoder(handle_unknown="ignore"), colunas_categoricas),
        ("numericas", "passthrough", colunas_numericas),
    ]
)
modelo_demanda = Pipeline(
    steps=[
        ("preparador", preparador),
        ("regressor_poisson", PoissonRegressor(alpha=0.5, max_iter=500)),
    ]
)
modelo_demanda.fit(treino[features], treino[alvo])
previsao_poisson = np.clip(modelo_demanda.predict(teste[features]), 0, None)

colunas_baseline = ["zone_id", "service_type", "weekday", "hour"]
medias_baseline = treino.groupby(colunas_baseline)[alvo].mean().to_dict()
chaves_teste = teste[colunas_baseline].itertuples(index=False, name=None)
previsao_baseline_sintetica = np.array([
    medias_baseline[tuple(chave)] for chave in chaves_teste
], dtype=float)

from sklearn.metrics import mean_poisson_deviance

def metricas_contagem(valores_reais, valores_previstos):
    reais = np.asarray(valores_reais, dtype=float)
    previstos = np.clip(np.asarray(valores_previstos, dtype=float), 1e-9, None)
    total_real = reais.sum()
    return {
        "mae": float(mean_absolute_error(reais, previstos)),
        "rmse": float(np.sqrt(mean_squared_error(reais, previstos))),
        "wape": float(np.abs(reais - previstos).sum() / total_real) if total_real else None,
        "desvio_poisson": float(mean_poisson_deviance(reais, previstos)),
    }

metricas_sinteticas = pd.DataFrame([
    {"metodo": "baseline_media_historica", **metricas_contagem(
        teste[alvo], previsao_baseline_sintetica
    )},
    {"metodo": "poisson_regressor", **metricas_contagem(
        teste[alvo], previsao_poisson
    )},
])
metricas_sinteticas["origem_dados"] = "sintetica"
metricas_sinteticas["observacoes_holdout"] = len(teste)
metricas_sinteticas["inicio_holdout"] = inicio_holdout
metricas_sinteticas.to_csv(
    RELATORIO_SINTETICO / "metricas_modelo_sintetico.csv",
    index=False,
    encoding="utf-8-sig",
)

previsoes_holdout_sinteticas = teste[
    ["timestamp", "zone_id", "service_type", "trips_requested", "trips_served", "unmet_requests"]
].copy()
previsoes_holdout_sinteticas["baseline_prediction"] = previsao_baseline_sintetica
previsoes_holdout_sinteticas["poisson_prediction"] = previsao_poisson
previsoes_holdout_sinteticas["is_synthetic"] = True
previsoes_holdout_sinteticas.to_csv(
    RELATORIO_SINTETICO / "previsoes_holdout_sintetico.csv",
    index=False,
    encoding="utf-8-sig",
)

print(f"Treino: {treino['timestamp'].min()} a {treino['timestamp'].max()}")
print(f"Holdout: {teste['timestamp'].min()} a {teste['timestamp'].max()}")
print(f"Observações de holdout: {len(teste):,}")
display(metricas_sinteticas)

### Comparação visual do holdout sintético

A seleção pelo menor WAPE serve apenas para escolher o método de demonstração neste experimento. Ela não qualifica o modelo para uma frota real.

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(13, 4))
metricas_sinteticas.set_index("metodo")["mae"].plot(
    kind="bar", ax=eixos[0], color=["#8b0b3d", "#385a7c"]
)
eixos[0].set_title("MAE no holdout sintético")
eixos[0].set_ylabel("Solicitações por hora")
eixos[0].tick_params(axis="x", rotation=20)
eixos[0].grid(axis="y", alpha=0.25)

resumo_holdout_diario = previsoes_holdout_sinteticas.set_index("timestamp").resample("D").sum(numeric_only=True)
resumo_holdout_diario[[
    "trips_requested", "baseline_prediction", "poisson_prediction"
]].plot(ax=eixos[1], marker="o", linewidth=1.5)
eixos[1].set_title("Observado sintético e previsões")
eixos[1].set_xlabel("Data sintética")
eixos[1].set_ylabel("Solicitações/dia")
eixos[1].grid(alpha=0.25)
fig.tight_layout()
fig.savefig(RELATORIO_SINTETICO / "avaliacao_modelo_sintetico.png", dpi=150, bbox_inches="tight")
plt.show()

## 11. Simular demanda e frota

Os cenários abaixo mantêm o calendário sintético e alteram apenas os parâmetros definidos: normal, pico de demanda e frota reduzida. Capacidade e solicitações não atendidas são recalculadas pelo gerador. A comparação representa consequências das hipóteses programadas.

In [ ]:
configuracoes_cenarios = [
    {"scenario": "normal", "demand": 1.0, "fleet": 1.0},
    {"scenario": "pico_demanda", "demand": 1.4, "fleet": 1.0},
    {"scenario": "frota_reduzida", "demand": 1.0, "fleet": 0.65},
]
partes_cenarios = []
resumos_cenarios = []
for configuracao in configuracoes_cenarios:
    dados_cenario = gerar_dados_sinteticos(
        dias=14,
        semente=SEMENTE_PADRAO,
        multiplicador_demanda=configuracao["demand"],
        multiplicador_frota=configuracao["fleet"],
        inicio="2025-03-01",
        cenario=configuracao["scenario"],
    )
    partes_cenarios.append(dados_cenario)
    pedidos = int(dados_cenario["trips_requested"].sum())
    atendidos = int(dados_cenario["trips_served"].sum())
    nao_atendidos = int(dados_cenario["unmet_requests"].sum())
    resumos_cenarios.append({
        "cenario": configuracao["scenario"],
        "fator_demanda": configuracao["demand"],
        "fator_frota": configuracao["fleet"],
        "solicitacoes_sinteticas": pedidos,
        "atendidas_pela_capacidade": atendidos,
        "nao_atendidas": nao_atendidos,
        "cobertura_simulada": atendidos / pedidos if pedidos else 0.0,
        "is_synthetic": True,
    })

dados_cenarios = pd.concat(partes_cenarios, ignore_index=True)
tabela_cenarios = pd.DataFrame(resumos_cenarios)
tabela_cenarios.to_csv(
    RELATORIO_SINTETICO / "resumo_cenarios_sinteticos.csv",
    index=False,
    encoding="utf-8-sig",
)
display(tabela_cenarios)

fig, eixos = plt.subplots(1, 2, figsize=(13, 4))
por_hora_cenario = (
    dados_cenarios.groupby(["scenario", "hour"])[
        ["trips_requested", "vehicles_available", "unmet_requests"]
    ].sum().reset_index()
)
for nome_cenario, grupo in por_hora_cenario.groupby("scenario"):
    eixos[0].plot(
        grupo["hour"], grupo["trips_requested"] / 14,
        marker="o", label=nome_cenario,
    )
eixos[0].set_title("Pedidos médios por hora nos cenários")
eixos[0].set_xlabel("Hora do dia")
eixos[0].set_ylabel("Pedidos simulados/hora (todas as zonas)")
eixos[0].legend()
eixos[0].grid(alpha=0.25)

tabela_cenarios.set_index("cenario")[[
    "solicitacoes_sinteticas", "atendidas_pela_capacidade", "nao_atendidas"
]].plot(kind="bar", ax=eixos[1], color=["#385a7c", "#4a8f63", "#b34747"])
eixos[1].set_title("Volume e capacidade simulados (14 dias)")
eixos[1].set_ylabel("Solicitações")
eixos[1].tick_params(axis="x", rotation=20)
eixos[1].grid(axis="y", alpha=0.25)
fig.tight_layout()
fig.savefig(RELATORIO_SINTETICO / "comparacao_cenarios.png", dpi=150, bbox_inches="tight")
plt.show()

## 12. Previsão demonstrativa e exportação do modelo

O PoissonRegressor será ajustado ao conjunto sintético de treino e salvo em joblib com um JSON de metadados. A previsão seguinte cobre 24 horas sintéticas para cada zona e serviço. O arquivo inclui a marca sintética e deve permanecer separado da previsão SIA/SUS.

In [ ]:
from itertools import product

proximo_inicio = df_modelo["timestamp"].max() + pd.Timedelta(hours=1)
horas_futuras = pd.date_range(proximo_inicio, periods=24, freq="h")
zonas = sorted(df_sintetico["zone_id"].unique())
servicos = sorted(df_sintetico["service_type"].unique())
previsao_futura = pd.DataFrame([
    {"timestamp": instante, "zone_id": zona, "service_type": servico}
    for instante, zona, servico in product(horas_futuras, zonas, servicos)
])
previsao_futura["hour"] = previsao_futura["timestamp"].dt.hour
previsao_futura["weekday"] = previsao_futura["timestamp"].dt.dayofweek
previsao_futura["is_weekend"] = (previsao_futura["weekday"] >= 5).astype("int8")
previsao_futura = adicionar_features_temporais(previsao_futura)
previsao_futura["poisson_prediction"] = np.clip(
    modelo_demanda.predict(previsao_futura[features]), 0, None
)
previsao_futura["scenario"] = "projecao_sintetica"
previsao_futura["is_synthetic"] = True
previsao_futura.to_csv(
    RELATORIO_SINTETICO / "previsao_sintetica_24h.csv",
    index=False,
    encoding="utf-8-sig",
)

melhor_metodo_sintetico = str(
    metricas_sinteticas.sort_values("wape").iloc[0]["metodo"]
)
pasta_modelos_sinteticos = ROOT / "models"
pasta_modelos_sinteticos.mkdir(parents=True, exist_ok=True)
CAMINHO_MODELO_SINTETICO = pasta_modelos_sinteticos / "moving_health_demanda_sintetica.joblib"
CAMINHO_METADADOS_SINTETICOS = pasta_modelos_sinteticos / "moving_health_demanda_sintetica.json"
joblib.dump(modelo_demanda, CAMINHO_MODELO_SINTETICO)

def normalizar_registro(valor):
    if isinstance(valor, (np.integer,)):
        return int(valor)
    if isinstance(valor, (np.floating,)):
        return float(valor)
    return valor

registro_metricas = [
    {chave: normalizar_registro(valor) for chave, valor in item.items()}
    for item in metricas_sinteticas.to_dict(orient="records")
]
metadados_modelo_sintetico = {
    "tipo_artefato": "experimento_sintetico",
    "modelo": "sklearn.pipeline.Pipeline com PoissonRegressor",
    "alvo": "trips_requested por hora × zona genérica × serviço",
    "origem_dados": "dados gerados por scripts/simulate_trip_demand.py",
    "semente": SEMENTE_PADRAO,
    "n_linhas": int(len(df_sintetico)),
    "periodo_treino_inicio": str(treino["timestamp"].min()),
    "periodo_treino_fim": str(treino["timestamp"].max()),
    "periodo_holdout_inicio": str(teste["timestamp"].min()),
    "periodo_holdout_fim": str(teste["timestamp"].max()),
    "metodo_menor_wape": melhor_metodo_sintetico,
    "metricas": registro_metricas,
    "versoes": runtime_versions,
    "aviso": "Métricas e previsões medem somente dados sintéticos; não demonstram desempenho em corridas reais.",
}
CAMINHO_METADADOS_SINTETICOS.write_text(
    json.dumps(metadados_modelo_sintetico, ensure_ascii=False, indent=2, default=str) + "\n",
    encoding="utf-8",
)
print(f"Modelo salvo: {CAMINHO_MODELO_SINTETICO.name}")
print(f"Método com menor WAPE na simulação: {melhor_metodo_sintetico}")
display(previsao_futura.head(6))

## 13. Baixar os resultados do Colab

O sistema de arquivos do Colab é temporário. Esta célula reúne o modelo SIA/SUS, o modelo sintético, os dois conjuntos de relatórios, o CSV sintético, previsões e gráficos em um ZIP para baixar antes de encerrar a sessão.

In [ ]:
if colab_files is None:
    print("Esta célula de download foi preparada para execução no Google Colab.")
else:
    from zipfile import ZIP_DEFLATED, ZipFile

    arquivos_resultado = [
        SAIDA_MODELO,
        SAIDA_PROCESSADA,
        CSV_SINTETICO,
        CAMINHO_MODELO_SINTETICO,
        CAMINHO_METADADOS_SINTETICOS,
        *SAIDA_RELATORIOS.glob("*.csv"),
        *RELATORIO_SINTETICO.glob("*"),
    ]
    pacote_resultados = Path.cwd() / "moving_health_resultados.zip"
    with ZipFile(pacote_resultados, "w", compression=ZIP_DEFLATED) as arquivo_zip:
        for arquivo in arquivos_resultado:
            arquivo_zip.write(
                arquivo,
                arcname=str(Path("moving_health_resultados") / arquivo.relative_to(ROOT)),
            )

    print("Pacote de resultados criado: moving_health_resultados.zip")
    colab_files.download(str(pacote_resultados))

## 14. Limitações e caminho para previsão real de viagens

- A extração é agregada por local de atendimento e não informa a origem dos pacientes.
- Não há relação observada entre a quantidade aprovada e pedidos de transporte do Moving Health.
- Os meses recentes do DATASUS podem ser atualizados.
- A métrica global dá mais peso a municípios com contagens maiores; devem ser avaliadas métricas por município e por faixa de volume antes de qualquer uso operacional.
- O candidato de regressão não superou o baseline no holdout.
- Nenhuma previsão deste notebook deve alocar motoristas, substituir profissionais ou determinar prioridade clínica.

Para prever viagens, o piloto deve registrar pedidos próprios com horário, área aproximada de origem/destino, categoria do serviço, estado do pedido e tempos operacionais. Depois será possível definir claramente se o alvo são pedidos iniciados, pedidos concluídos ou necessidade de cobertura e fazer validação temporal com dados próprios.

Consulte README.md para as instruções de execução e docs/synthetic_data.md para as hipóteses da simulação.